In [1]:
import pandas as pd
from sklearn import preprocessing
import ast

In [2]:
class preprocessor():
    def __init__(self,data_path,keywords_limit):
        self.df = pd.read_csv(data_path,index_col=False)
        self.df['keywords'] = self.df['keywords'].apply(lambda x: ast.literal_eval(x)[:keywords_limit])

        self.keywords_list = self.df['keywords'].to_list()

        self.keywords_list = list(set([j for sub in self.keywords_list for j in sub]))
        self.keyword_enocder = dict(zip(range(len(self.keywords_list)),self.keywords_list))
        self.keyword_to_label_enocder = dict(zip(self.keywords_list,range(len(self.keywords_list))))
        self.user_index_mapping, self.url_index_mapping, self.label_index_mapping,self.index_user_mapping,self.index_url_mapping,self.index_label_mapping = self.label_ecoder()

        article_list = self.df[['keywords','label_index','Dominant_Topic']].values.tolist()
        url_index_label_list = self.df[['url_index','label_index','Dominant_Topic']].values.tolist()
        
        self.url_to_keyword_and_labels_dict = dict(zip(self.df.url,pd.Series(article_list)))
        self.url_to_index_and_labels_dict = dict(zip(self.df.url,pd.Series(url_index_label_list)))

        self.url_to_keyword_dict = dict(zip(self.df.url_index,self.df.keywords))
        self.keyword_to_url_dict = dict((str(v), k) for k, v in self.url_to_keyword_dict.items())

        self.user_df = self.df[['user_index']]
        self.url_and_label_df = self.df[['url_index','label_index','Dominant_Topic']]

        self.user_flds = [n for n in self.user_df.columns.values]
        self.url_flds = [n for n in self.url_and_label_df.columns.values]
        self.y = self.df[['final_rating']]

        self.user_embedding_cardinality = {n: len(c.astype('category').cat.categories)+1 for n,c in self.df[['user_index']].items()}
        self.url_embedding_cardinality = {'keywords': len(self.keywords_list)+1,'label': len(self.df.label_index.unique()),'Dominant_Topic': len(self.df.Dominant_Topic.unique()) }

        self.user_emb_sizes, self.url_emb_sizes = self.get_user_url_label_embedding()


    def get_keyword_label(self,keyword_list):
        keyword_to_label = []
        for keyword in keyword_list:
            keyword_to_label.append(self.keyword_to_label_enocder[keyword])
        return keyword_to_label

    def label_ecoder(self,):
        user_le = preprocessing.LabelEncoder()
        url_le = preprocessing.LabelEncoder()
        label_le = preprocessing.LabelEncoder()
        self.df['user_index'] = user_le.fit_transform(self.df['user'])
        self.df['url_index'] = url_le.fit_transform(self.df['url'])
        self.df['label_index'] = label_le.fit_transform(self.df['label'])
        user_le_mapping = dict(zip(user_le.classes_, user_le.transform(user_le.classes_)))
        url_le_mapping = dict(zip(url_le.classes_, url_le.transform(url_le.classes_)))
        label_le_mapping = dict(zip(label_le.classes_, label_le.transform(label_le.classes_)))

        le_user_mapping = dict(zip(user_le.transform(user_le.classes_),user_le.classes_))
        le_url_mapping = dict(zip(url_le.transform(url_le.classes_),url_le.classes_))
        label_le_mapping = dict(zip(label_le.transform(label_le.classes_), label_le.classes_))

        self.df['Dominant_Topic'] = self.df['Dominant_Topic'].apply(lambda x : int(x))
        self.df['keywords'] = self.df['keywords'].apply(lambda x: self.get_keyword_label(x))

        return user_le_mapping,url_le_mapping,label_le_mapping,le_user_mapping,le_url_mapping,label_le_mapping

    def get_user_url_label_embedding(self):
        user_emb_sizes = [(size, 108) for item, size in self.user_embedding_cardinality.items()]
        url_emb_sizes = []
        for item, size in self.url_embedding_cardinality.items():
            if item == 'keywords':
                temp_tuple = tuple([size,108])
            elif item == 'label':
                temp_tuple = tuple([size,int(size//2.71828)])
            elif item == 'Dominant_Topic':
                temp_tuple = tuple([size,int(size//2.71828)])
            url_emb_sizes.append(temp_tuple)
        return user_emb_sizes,url_emb_sizes

In [4]:
data_path = 'train_data/train.csv'

print('----------Preprocessing(loading data)--------------------')
keyword_lim = 20
pre = preprocessor(data_path,keywords_limit = keyword_lim)
print('----------Preprocessing Completed--------------------')
dropout = 0.01
learning_rate = 0.001
print("finish loading Data...)")

----------Preprocessing(loading data)--------------------
----------Preprocessing Completed--------------------
finish loading Data...)


In [5]:
pre.df

,user,url,additional_images,content,date,head_image,heading,label,source,author,author_img,all_content,all_heading_content_list,all_heading_content,keywords,Dominant_Topic,final_rating,user_index,url_index,label_index
0,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
1,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
2,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
3,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
4,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707192,5cb16210f5bc5f18bb852ab846f3b5a2,https://www.thahakhabar.com/news/127679,[],['\xa0नेकपा एमालेका अध्यक्ष एवं प्रधानमन्त्री ...,2078-02-24 00:00:00+05:45,https://thahacdn.prixacdn.net/media/albums/Kha...,खनाल-नेपाल समूहका नेता सरकारमा सहभागी हुँदै !,राजनीति,thahakhabar,थाहा संवाददाता,https://thahacdn.prixacdn.net/media/Logo_Thaha...,खनाल-नेपाल समूहका नेता सरकारमा सहभागी हुँदै ! ...,"['खनाल-नेपाल', 'समूह', 'नेता', 'सरकार', 'सहभाग...",खनाल-नेपाल समूह नेता सरकार सहभागी नेकपा एमाल अ...,"[13635, 1725, 19919, 25420, 8942, 5742, 31687,...",11,2.876623,67476,35675,22
707193,93c95988ec06dc1569a02ccb931c43a4,https://www.thahakhabar.com/news

In [8]:
import pickle

def load_pkl(pkl_file):
    with open(pkl_file, 'rb') as inp:
        pickled_obj = pickle.load(inp)
    return pickled_obj

def save_pkl(obj, filename):
    with open(filename, 'wb') as outp:  # Overwrites any existing file.
        pickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)



In [10]:
save_pkl(pre, 'train_data/pre.pkl')

In [11]:
pre.df.columns

Index(['user', 'url', 'additional_images', 'content', 'date', 'head_image',
       'heading', 'label', 'source', 'author', 'author_img', 'all_content',
       'all_heading_content_list', 'all_heading_content', 'keywords',
       'Dominant_Topic', 'final_rating', 'user_index', 'url_index',
       'label_index'],
      dtype='object')

In [13]:
pre.df.head(2)

,user,url,additional_images,content,date,head_image,heading,label,source,author,author_img,all_content,all_heading_content_list,all_heading_content,keywords,Dominant_Topic,final_rating,user_index,url_index,label_index
0,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
1,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2


In [14]:
p = load_pkl('train_data/pre.pkl')

In [15]:
p.df

,user,url,additional_images,content,date,head_image,heading,label,source,author,author_img,all_content,all_heading_content_list,all_heading_content,keywords,Dominant_Topic,final_rating,user_index,url_index,label_index
0,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
1,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
2,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
3,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
4,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,https://ratopatis.prixacdn.net/media/Untitled-...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,"['यस्तो', 'विनिमयदरः', 'अमेरि', 'कतारी', 'रिया...",यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707192,5cb16210f5bc5f18bb852ab846f3b5a2,https://www.thahakhabar.com/news/127679,[],['\xa0नेकपा एमालेका अध्यक्ष एवं प्रधानमन्त्री ...,2078-02-24 00:00:00+05:45,https://thahacdn.prixacdn.net/media/albums/Kha...,खनाल-नेपाल समूहका नेता सरकारमा सहभागी हुँदै !,राजनीति,thahakhabar,थाहा संवाददाता,https://thahacdn.prixacdn.net/media/Logo_Thaha...,खनाल-नेपाल समूहका नेता सरकारमा सहभागी हुँदै ! ...,"['खनाल-नेपाल', 'समूह', 'नेता', 'सरकार', 'सहभाग...",खनाल-नेपाल समूह नेता सरकार सहभागी नेकपा एमाल अ...,"[13635, 1725, 19919, 25420, 8942, 5742, 31687,...",11,2.876623,67476,35675,22
707193,93c95988ec06dc1569a02ccb931c43a4,https://www.thahakhabar.com/news

In [16]:
a = "['काठमाडौं उपत्यकाको आकाशमा मंगलवार दिउँसो एक्कासि धेरै हेलिकप्टर देखिएका छन् ।', 'नेपाली सेनाले शुक्रवार सेना दिवस तथा महाशिवरात्रिको अवसरमा गर्ने प्रदर्शनको पूर्वाभ्यासका क्रममा हेलिकप्टर उडाइएको हो ।', 'नेपाली सेनाका प्रधान सेनापति पूर्णचन्द्र थापाको नेतृत्वमा शिवरात्रिमा हुने सबै प्रकारका प्रदर्शन र देखाइने कवाजहरूको आज अन्तिम पूर्वाभ्यास गरिरहेको सैनिक जनसम्पर्क निर्देशनालयले\\xa0 जनाएको छ ।', 'निर्देशनालयका अनुसार शिवरात्रि\\xa0तथा सेना दिवसका अवसरका देखाइने कवाज, हेलिकप्टर तथा विभिन्न प्रकारका संस्कृतिका झाँकीहरूको पूर्वाभ्यास गरिरहेको छ ।', 'राष्ट्रपति सवारी, प्रधानमन्त्री सवारीलगायत भीभीआईपी सवारीको पनि पूर्वाभ्यास गरिएको सेनाले जनाएको छ ।', ' \\xa0']"
ast.literal_eval(a)

['काठमाडौं उपत्यकाको आकाशमा मंगलवार दिउँसो एक्कासि धेरै हेलिकप्टर देखिएका छन् ।',
 'नेपाली सेनाले शुक्रवार सेना दिवस तथा महाशिवरात्रिको अवसरमा गर्ने प्रदर्शनको पूर्वाभ्यासका क्रममा हेलिकप्टर उडाइएको हो ।',
 'नेपाली सेनाका प्रधान सेनापति पूर्णचन्द्र थापाको नेतृत्वमा शिवरात्रिमा हुने सबै प्रकारका प्रदर्शन र देखाइने कवाजहरूको आज अन्तिम पूर्वाभ्यास गरिरहेको सैनिक जनसम्पर्क निर्देशनालयले\xa0 जनाएको छ ।',
 'निर्देशनालयका अनुसार शिवरात्रि\xa0तथा सेना दिवसका अवसरका देखाइने कवाज, हेलिकप्टर तथा विभिन्न प्रकारका संस्कृतिका झाँकीहरूको पूर्वाभ्यास गरिरहेको छ ।',
 'राष्ट्रपति सवारी, प्रधानमन्त्री सवारीलगायत भीभीआईपी सवारीको पनि पूर्वाभ्यास गरिएको सेनाले जनाएको छ ।',
 ' \xa0']